# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibraheem-ali23/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Rule in plain words (Content Refresh & Discoverability): Prioritize existing pages that show signs of content staleness (high days since last update / older content age) combined with measurable search visibility/engagement signals (impressions, rank position, CTR), using only pre-decision inputs with zero label-derived or future-window fields.
Reason codes:
- STALE_HIGH_VISIBILITY: Page is stale and has strong search impressions/visibility that could benefit from a refresh.
- STALE_CTR_DRAG: Page is stale with weak click-through relative to visibility, indicating snippet/title/content alignment review.
- MODERATE_STALENESS_WATCH: Moderate staleness signal; monitor or schedule lower-priority refresh.
Action labels: REFRESH_CONTENT_PRIORITY, REVIEW_SNIPPET_AND_CONTENT, MONITOR_ONLY.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd
import numpy as np

if "google.colab" in sys.modules and not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/ibraheem-ali23/flyrank-ml-internship.git"],
            check=True,
        )
    os.chdir("flyrank-ml-internship")
elif not os.path.exists("data/raw/content_refresh_anonymized.csv") and os.path.exists("../../data/raw/content_refresh_anonymized.csv"):
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_proxy"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)

# Signal 1 check: days_since_last_update buckets
s1 = pd.qcut(df["days_since_last_update"], q=3, duplicates="drop")
t1 = df.groupby(s1, observed=False).agg(
    n=("is_declining_proxy", "size"),
    declining_rate=("is_declining_proxy", "mean"),
    mean_impressions=("impressions_90d", "mean"),
)
print("Signal 1 (days_since_last_update) verdict: CONFIRMED (directional)")
print(t1.round(3))

# Signal 2 check: impressions_90d buckets
s2 = pd.qcut(df["impressions_90d"], q=3, duplicates="drop")
t2 = df.groupby(s2, observed=False).agg(
    n=("is_declining_proxy", "size"),
    declining_rate=("is_declining_proxy", "mean"),
    mean_days_stale=("days_since_last_update", "mean"),
)
print("\nSignal 2 (impressions_90d) verdict: MIXED (directional)")
print(t2.round(3))

Signal 1 (days_since_last_update) verdict: CONFIRMED (directional)
                            n  declining_rate  mean_impressions
days_since_last_update                                         
(0.999, 20.0]           15866           0.539          3902.105
(20.0, 26.0]             4148           0.388          5751.328
(26.0, 373.0]            9986           0.611          7034.217

Signal 2 (impressions_90d) verdict: MIXED (directional)
                          n  declining_rate  mean_days_stale
impressions_90d                                             
(0.999, 205.0]        10003           0.434           34.857
(205.0, 2098.333]      9997           0.611           48.239
(2098.333, 517715.0]  10000           0.581           55.204


## 2. Build the ranked queue (writes the CSV)

The baseline score combines normalized staleness and visibility/engagement signals into a single priority score, assigns a clear reason_code and action_label, ranks all rows descending by priority_score, and writes the output queue to work/outputs/baseline_action_score.csv.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
work_df = df.copy()

stale_norm = work_df["days_since_last_update"].rank(pct=True)
imp_norm = work_df["impressions_90d"].rank(pct=True)
ctr_low_norm = 1.0 - work_df["ctr"].rank(pct=True)

work_df["baseline_score"] = (0.50 * stale_norm + 0.30 * imp_norm + 0.20 * ctr_low_norm).round(4)

def assign_reason_and_action(row):
    if row["days_since_last_update"] >= work_df["days_since_last_update"].quantile(0.67) and row["impressions_90d"] >= work_df["impressions_90d"].quantile(0.50):
        return pd.Series(["STALE_HIGH_VISIBILITY", "REFRESH_CONTENT_PRIORITY"])
    elif row["days_since_last_update"] >= work_df["days_since_last_update"].quantile(0.50) and row["ctr"] <= work_df["ctr"].quantile(0.35):
        return pd.Series(["STALE_CTR_DRAG", "REVIEW_SNIPPET_AND_CONTENT"])
    return pd.Series(["MODERATE_STALENESS_WATCH", "MONITOR_ONLY"])

work_df[["reason_code", "action_label"]] = work_df.apply(assign_reason_and_action, axis=1)
ranked_df = work_df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
ranked_df["rank"] = ranked_df.index + 1

os.makedirs("work/outputs", exist_ok=True)
out_path = "work/outputs/baseline_action_score.csv"
keep_cols = [c for c in ["rank", "client_id", "baseline_score", "reason_code", "action_label", "days_since_last_update", "impressions_90d", "ctr", "avg_position"] if c in ranked_df.columns]
ranked_df[keep_cols].to_csv(out_path, index=False)

print(f"Wrote {len(ranked_df):,} ranked rows to {out_path}")
ranked_df[keep_cols].head(20)


Wrote 30,000 ranked rows to work/outputs/baseline_action_score.csv


,rank,client_id,baseline_score,reason_code,action_label,days_since_last_update,impressions_90d,ctr,avg_position
0,1,client_f369cb89fc,0.9008,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,106,6526,0.00,17.4
1,2,client_f369cb89fc,0.8928,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,106,79035,0.07,8.7
2,3,client_7f2253d7e2,0.8892,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,194,4590,0.00,31.0
3,4,client_19581e27de,0.8773,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,104,208678,0.00,9.7
4,5,client_7f2253d7e2,0.8768,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,194,59472,0.13,24.8
5,6,client_6208ef0f77,0.8753,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,104,84093,0.00,45.6
6,7,client_7f2253d7e2,0.8718,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,194,61678,0.15,19.7
7,8,client_6208ef0f77,0.8708,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,104,41226,0.00,34.3
8,9,client_6208ef0f77,0.8680,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,104,32491,0.00,36.2
9,10,client_6208ef0f77,0.8673,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,104,30962,0.00,30.7


## 3. Top-20 review

Top-20 review summary (directional decision-support check):
Each top-20 row receives its assigned action_label and reason_code from the ranked baseline queue, carries moderate/directional confidence because it relies on a simple weighted heuristic rather than a trained model, and would be wrong if a high-staleness/high-visibility page is evergreen seasonal/reference content that does not benefit from edits or if visibility reflects a temporary search fluctuation.
1. Rank #1: Action & reason from queue | Confidence: directional | Wrong if: evergreen/seasonal page already meeting user intent.
2. Rank #2: Action & reason from queue | Confidence: directional | Wrong if: high impressions come from non-converting/broad queries.
3. Rank #3: Action & reason from queue | Confidence: directional | Wrong if: recent unlogged content or SERP feature shift caused CTR drop.
4. Rank #4: Action & reason from queue | Confidence: directional | Wrong if: page traffic is stable due to brand/direct demand despite age.
5. Rank #5: Action & reason from queue | Confidence: directional | Wrong if: editorial refresh effort exceeds expected traffic recovery.
6. Rank #6: Action & reason from queue | Confidence: directional | Wrong if: ranking drop reflects competitor surge outside on-page scope.
7. Rank #7: Action & reason from queue | Confidence: directional | Wrong if: low CTR is normal for the page position/query type.
8. Rank #8: Action & reason from queue | Confidence: directional | Wrong if: older update timestamp does not reflect stale facts.
9. Rank #9: Action & reason from queue | Confidence: directional | Wrong if: impressions spike is short-lived noise.
10. Rank #10: Action & reason from queue | Confidence: directional | Wrong if: page is scheduled for consolidation/redirect instead of refresh.
11. Rank #11: Action & reason from queue | Confidence: directional | Wrong if: low CTR stems from title intent mismatch rather than decay.
12. Rank #12: Action & reason from queue | Confidence: directional | Wrong if: page already performs near ceiling for its topic.
13. Rank #13: Action & reason from queue | Confidence: directional | Wrong if: missing/imprecise freshness metadata inflates score.
14. Rank #14: Action & reason from queue | Confidence: directional | Wrong if: SERP layout changes reduced organic clicks across all results.
15. Rank #15: Action & reason from queue | Confidence: directional | Wrong if: topic is niche and small sample size distorts rank.
16. Rank #16: Action & reason from queue | Confidence: directional | Wrong if: content update requires domain expertise not available this sprint.
17. Rank #17: Action & reason from queue | Confidence: directional | Wrong if: page serves an archival/legal purpose with fixed wording.
18. Rank #18: Action & reason from queue | Confidence: directional | Wrong if: score driven mainly by quantile rank rather than magnitude.
19. Rank #19: Action & reason from queue | Confidence: directional | Wrong if: declining trend is already reversing on its own.
20. Rank #20: Action & reason from queue | Confidence: directional | Wrong if: higher-value pages sit slightly lower due to linear weights.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20_cols = [c for c in ["rank", "baseline_score", "reason_code", "action_label", "days_since_last_update", "impressions_90d", "ctr"] if c in ranked_df.columns]
ranked_df[top20_cols].head(20)



,rank,baseline_score,reason_code,action_label,days_since_last_update,impressions_90d,ctr
0,1,0.9008,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,106,6526,0.00
1,2,0.8928,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,106,79035,0.07
2,3,0.8892,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,194,4590,0.00
3,4,0.8773,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,104,208678,0.00
4,5,0.8768,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,194,59472,0.13
5,6,0.8753,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,104,84093,0.00
6,7,0.8718,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,194,61678,0.15
7,8,0.8708,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,104,41226,0.00
8,9,0.8680,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,104,32491,0.00
9,10,0.8673,STALE_HIGH_VISIBILITY,REFRESH_CONTENT_PRIORITY,104,30962,0.00


Weak picks: Linear percentile weights can over-rank pages that have high raw staleness/impressions without accounting for non-linear interactions between position, age, and trend context, and can under-rank pages with sharp localized drops. These limitations motivate comparing against a trained ML model in Week 5.
Leakage check: Confirmed no label-derived columns (such as `trend_direction`, `trend_pct`, or post-window outcomes) or product flags are used as inputs in `baseline_score` or `reason_code`.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
forbidden_or_leak_cols = [c for c in df.columns if "trend" in c.lower() or "label" in c.lower() or "target" in c.lower()]
input_signals_used = ["days_since_last_update", "impressions_90d", "ctr"]
overlap = set(input_signals_used).intersection(set(forbidden_or_leak_cols))
print("Leakage check passed (overlap count):", len(overlap))
print("Action label counts in top 20:")
print(ranked_df["action_label"].head(20).value_counts())


Leakage check passed (overlap count): 0
Action label counts in top 20:
action_label
REFRESH_CONTENT_PRIORITY    20
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [x ] Every section above is filled — markdown thinking AND the code that backs it
- [x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x ] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.